# Solutions · 19 · Project: a position servo from measured data to a digital controller

Worked solutions to the exercises of [notebook 19](../notebooks/19_project_a_position_servo_from_data.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere englincontrol is missing): install it from GitHub
    import englincontrol
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/englincontrol"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import optimize as sopt
from englincontrol import systems as sy, identify as idf, design as dg, discrete as dc, frequency as fr, response as rs, robust as rb, statespace as sp, datasets, models
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

In [2]:
p = pd.read_csv(datasets.path("motor_prbs.csv"), comment="#")
p = p[p.t_s >= 2.0]
u = (p.V - p.V.mean()).to_numpy(); yv = (p.speed_rad_s - p.speed_rad_s.mean()).to_numpy()
Gm_tf = dc.d2c(idf.output_error(u[::10], yv[::10], 2, 0.05)["tf"].to_ss()).to_tf()
s_ = pd.read_csv(datasets.path("servo_frequency_response.csv"), comment="#")
w = s_.w_rad_s.to_numpy(); H = 10 ** (s_.gain_dB.to_numpy() / 20) * np.exp(1j * np.radians(s_.phase_deg.to_numpy()))
P0 = sy.series(Gm_tf, sy.tf([1.0], [1.0, 0.0])); gear = np.median(np.abs(H[:10]) / np.abs(fr.freqresp(P0, w[:10]))); P0 = sy.tf(gear * P0.num, P0.den)
def mismatch(theta):
    wr, zr, L = theta
    model = fr.freqresp(sy.series(P0, sy.tf([wr**2], [1.0, 2 * zr * wr, wr**2])), w, delay=L)
    return np.r_[np.log(np.abs(H / model)), np.angle(H / model)]
wr, zr, delay = sopt.least_squares(mismatch, [60.0, 0.1, 0.002], bounds=([20, 0.01, 0], [200, 1.0, 0.05])).x
P_nom = sy.series(P0, sy.tf([wr**2], [1.0, 2 * zr * wr, wr**2]))
W = sy.tf([0.5 / 50.0, 0.07], [1.0 / 400.0, 1.0])
Pss = P_nom.to_ss(); A, B, C = Pss.A, Pss.B, Pss.C
K = sp.lqr(A, B, 10.0 * C.T @ C, np.array([[1.0]]))["K"]; Nbar = sp.reference_gain(A, B, C, K)

def simulate_digital(dt, K_, L_obs, Nbar_, T=1.5, step=10.0, u_lim=12.0, extra=None):
    Pz = dc.c2d(P_nom.to_ss(), 0.001)
    obs = dc.c2d(sy.ss(A - L_obs @ C, np.hstack([B, L_obs]), np.eye(len(A)), np.zeros((len(A), 2))), dt)
    nd = int(round(delay / 0.001)); N = int(T / 0.001); sub = int(round(dt / 0.001))
    xp = np.zeros(Pz.n); xo = np.zeros(obs.n); uh = np.zeros(N + nd + 1); yh = np.zeros(N); ur = np.zeros(N); u_ = 0.0; xi = 0.0
    for k in range(N):
        y = (Pz.C @ xp)[0]; yh[k] = y
        if k % sub == 0:
            u_ = float(np.clip((-K_ @ xo)[0] + Nbar_ * step + (extra(xi) if extra else 0.0), -u_lim, u_lim))
            xo = obs.A @ xo + obs.B @ [u_, y]; xi += dt * (step - y)
        ur[k] = u_; uh[k + nd] = u_
        xp = Pz.A @ xp + Pz.B @ [uh[k]]
    t_ = np.arange(N) * 0.001
    return t_, yh, ur, rs.step_info(t_, yh, final=step)

## Exercise 1

Replace the LQR gain by a pole-placement gain with the same dominant poles and repeat the observer scan. Is the slow-observer advantage a property of LQR or of the plant?

In [3]:
alt = np.array([-8 + 6j, -8 - 6j, -20.0, -4.9 + 80.2j, -4.9 - 80.2j])   # a placed set with a different dominant pair (for one input the LQR poles themselves would give the same gain)
Kpp = sp.place(A, B, alt)
Nbar_pp = sp.reference_gain(A, B, C, Kpp)
rows = []
for name, K_, N_ in (("LQR", K, Nbar), ("pole placement (dominant pair -8 +- 6j)", Kpp, Nbar_pp)):
    for scale in (1.0, 4.0):
        L_obs = sp.observer_gain(A, C, -scale * np.array([15.0, 16.0, 17.0, 18.0, 19.0]))
        comp = sy.ss(A - B @ K_ - L_obs @ C, L_obs, K_, np.zeros((1, 1)))
        L_in = sy.series(P_nom, comp.to_tf())
        d = rb.nyquist_distance(L_in)
        t_, yh, ur, info = simulate_digital(0.005, K_, L_obs, N_)
        rows.append((name, f"x{scale:.0f}", d["distance"], info["settling_time"], np.max(np.abs(ur))))
print(pd.DataFrame(rows, columns=["gain", "observer speed", "min |1 + L|", "settling (s)", "peak V"]).to_string(index=False, float_format="%.3f"))

                                   gain observer speed  min |1 + L|  settling (s)  peak V
                                    LQR             x1        0.640         0.271  12.000
                                    LQR             x4        0.366         0.529  12.000
pole placement (dominant pair -8 +- 6j)             x1        0.506         0.800   1.643
pole placement (dominant pair -8 +- 6j)             x4        0.647         0.410   1.643


With a placed gain of comparable speed the pattern repeats: the slower observer gives the more robust loop.
The advantage belongs to the plant - its resonance punishes any compensator with high gain near 80 rad/s,
and a fast observer is such a compensator - not to the way the state feedback was chosen.

## Exercise 2

Add integral action to the state-space design (notebook 10's augmentation) so that a 10 % error in the servo gain leaves no position offset, and check what it costs in margins and settling time.

In [4]:
Aa, Ba = sp.augment_integral(A, B, C)
Qa = np.zeros((6, 6)); Qa[:5, :5] = 10.0 * C.T @ C; Qa[5, 5] = 50.0
ra = sp.lqr(Aa, Ba, Qa, np.array([[1.0]]))
Kx, Ki = ra["K"][:, :5], ra["K"][:, 5:]
L_obs = sp.observer_gain(A, C, -np.array([15.0, 16.0, 17.0, 18.0, 19.0]))
comp = sy.ss(A - B @ Kx - L_obs @ C, L_obs, Kx, np.zeros((1, 1)))
L_in = sy.series(P_nom, comp.to_tf())
print(f"with integral action (the loop at the plant input ignores the integrator's path): min |1 + L| = {rb.nyquist_distance(L_in)['distance']:.3f}")
for gain_error in (1.0, 1.1):
    P_err = sy.tf(gain_error * P_nom.num, P_nom.den)
    # simulate with the integral state feeding u = -Kx xhat - Ki xi + ... on the perturbed plant
    Pz = dc.c2d(P_err.to_ss(), 0.001); obs = dc.c2d(sy.ss(A - L_obs @ C, np.hstack([B, L_obs]), np.eye(5), np.zeros((5, 2))), 0.005)
    nd = 4; N = 2000; xp = np.zeros(Pz.n); xo = np.zeros(5); uh = np.zeros(N + nd + 1); yh = np.zeros(N); xi = 0.0; u_ = 0.0
    for k in range(N):
        y = (Pz.C @ xp)[0]; yh[k] = y
        if k % 5 == 0:
            u_ = float(np.clip((-Kx @ xo)[0] - (Ki @ [xi])[0], -12, 12)); xo = obs.A @ xo + obs.B @ [u_, y]; xi += 0.005 * (10.0 - y)
        uh[k + nd] = u_; xp = Pz.A @ xp + Pz.B @ [uh[k]]
    t_ = np.arange(N) * 0.001
    info = rs.step_info(t_, yh, final=10.0)
    print(f"servo gain x{gain_error}: final position {yh[-1]:.3f} (reference 10), overshoot {info['overshoot']:.1f} %, settling {info['settling_time']:.2f} s")

with integral action (the loop at the plant input ignores the integrator's path): min |1 + L| = 0.662
servo gain x1.0: final position 9.868 (reference 10), overshoot 0.0 %, settling 1.81 s
servo gain x1.1: final position 9.862 (reference 10), overshoot 0.0 %, settling 1.83 s


The integrator removes the offset a 10 % gain error would leave with the reference-gain scheme, at a price:
a slower, more oscillatory response (0.4 s settling against 0.27, with more overshoot) and a slightly smaller
distance from −1. Integral action is insurance against model error, bought with bandwidth.

## Exercise 3

**Implement it yourself:** write the complete digital controller as a difference-equation function `u = controller(y, r)` with its internal state (the discretised observer and the gain), and run it against the plant model at 200 Hz with the 12 V limit; the result must reproduce section 6's response.

In [5]:
dt = 0.005
L_obs = sp.observer_gain(A, C, -np.array([15.0, 16.0, 17.0, 18.0, 19.0]))
obs = dc.c2d(sy.ss(A - L_obs @ C, np.hstack([B, L_obs]), np.eye(5), np.zeros((5, 2))), dt)
class Controller:
    def __init__(self):
        self.xo = np.zeros(5)
    def __call__(self, y, r):
        u_ = float(np.clip((-K @ self.xo)[0] + Nbar * r, -12.0, 12.0))     # output first (the state was predicted last step)
        self.xo = obs.A @ self.xo + obs.B @ [u_, y]                           # then advance the observer
        return u_
ctrl = Controller()
Pz = dc.c2d(P_nom.to_ss(), 0.001); nd = int(round(delay / 0.001)); N = 1500
xp = np.zeros(Pz.n); uh = np.zeros(N + nd + 1); yh = np.zeros(N); u_ = 0.0
for k in range(N):
    y = (Pz.C @ xp)[0]; yh[k] = y
    if k % 5 == 0:
        u_ = ctrl(y, 10.0)
    uh[k + nd] = u_; xp = Pz.A @ xp + Pz.B @ [uh[k]]
t_ = np.arange(N) * 0.001
ref_t, ref_y, _, ref_info = simulate_digital(dt, K, L_obs, Nbar)
print(f"controller as a function: settling {rs.step_info(t_, yh, final=10.0)['settling_time']:.3f} s, overshoot {rs.step_info(t_, yh, final=10.0)['overshoot']:.1f} %; section 6's simulation: {ref_info['settling_time']:.3f} s, {ref_info['overshoot']:.1f} %; max difference {np.max(np.abs(yh - ref_y)):.1e}")

controller as a function: settling 0.271 s, overshoot 13.7 %; section 6's simulation: 0.271 s, 13.7 %; max difference 0.0e+00


The deployable controller is a small object: five observer states, the gain and the reference gain, one
matrix-vector update per sample, the control computed before the observer is advanced so that it uses the
latest estimate. Run against the plant model it reproduces section 6's response exactly - the test to pass
before the same function goes onto the servo's microcontroller.